# Feature Selection for Phishing URL Detection

## 1. Introduction

Feature selection is the process of identifying the most useful predictive features from the available feature set while removing irrelevant, redundant, or low-information features.

In the previous notebook, feature engineering expanded the dataset from 50 original numerical predictive features to 64 predictive features by creating meaningful URL, ratio, density, and webpage structural features.

Feature selection is now required to reduce unnecessary complexity and retain features that provide useful information for phishing URL classification.

## 2. Objectives

The main objectives of this notebook are:

- Load the engineered dataset.
- Identify the complete predictive feature set.
- Separate predictors from the target variable.
- Analyze feature variance.
- Analyze feature correlation and redundancy.
- Measure the relationship between features and the target.
- Select the most informative features.
- Avoid target leakage during feature selection.
- Prepare the final selected feature set for model building.

## 3. Feature Selection Strategy

The following techniques will be considered:

### A. Variance Analysis

Features with extremely low variance provide very little information because their values remain almost unchanged across observations.

### B. Correlation Analysis

Highly correlated features may contain similar information. Redundant features can increase model complexity without providing significant additional information.

### C. Target Relationship

The relationship between each predictive feature and the target variable will be examined to identify features that contain useful classification information.

### D. Final Feature Selection

The final feature set will be created by combining statistical analysis with domain relevance.

The selection process will avoid blindly removing features only because they are correlated. Features will be retained when they provide meaningful and complementary information.

## 4. Leakage Prevention

The target variable `label` will not be used as an input feature.

Feature-selection decisions that depend on the target will be calculated using the training data only. The test data will remain completely unseen during feature selection.

The URL-group splitting strategy from the preprocessing stage will be maintained to prevent the same URL from appearing in both training and testing datasets.

## 5. Expected Outcome

At the end of this notebook, a final subset of informative predictive features will be selected from the 64 engineered predictive features.

The selected features will be used in the subsequent model-building stage for training and evaluating phishing URL classification models.

In [1]:
# Import libraries and load the engineered dataset

import pandas as pd
import numpy as np
import os

dataset_path = "../dataset/PhiUSIIL_Engineered_Features.csv"

df = pd.read_csv(dataset_path)

print("FEATURE SELECTION ENVIRONMENT READY")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

FEATURE SELECTION ENVIRONMENT READY
Dataset shape: (235795, 70)
Number of records: 235795
Number of columns: 70


In [2]:
# Identify predictive features

target_column = "label"

excluded_columns = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "Title",
    target_column
]

# Select numerical predictive features
predictive_features = [
    col for col in df.columns
    if col not in excluded_columns
    and pd.api.types.is_numeric_dtype(df[col])
]

print("PREDICTIVE FEATURE IDENTIFICATION")
print("=" * 60)

print("Total predictive features:", len(predictive_features))

print("\nExcluded columns:")
for col in excluded_columns:
    print("-", col)

print("\nPredictive feature list:")
for i, feature in enumerate(predictive_features, start=1):
    print(f"{i:2d}. {feature}")

PREDICTIVE FEATURE IDENTIFICATION
Total predictive features: 64

Excluded columns:
- FILENAME
- URL
- Domain
- TLD
- Title
- label

Predictive feature list:
 1. URLLength
 2. DomainLength
 3. IsDomainIP
 4. URLSimilarityIndex
 5. CharContinuationRate
 6. TLDLegitimateProb
 7. URLCharProb
 8. TLDLength
 9. NoOfSubDomain
10. HasObfuscation
11. NoOfObfuscatedChar
12. ObfuscationRatio
13. NoOfLettersInURL
14. LetterRatioInURL
15. NoOfDegitsInURL
16. DegitRatioInURL
17. NoOfEqualsInURL
18. NoOfQMarkInURL
19. NoOfAmpersandInURL
20. NoOfOtherSpecialCharsInURL
21. SpacialCharRatioInURL
22. IsHTTPS
23. LineOfCode
24. LargestLineLength
25. HasTitle
26. DomainTitleMatchScore
27. URLTitleMatchScore
28. HasFavicon
29. Robots
30. IsResponsive
31. NoOfURLRedirect
32. NoOfSelfRedirect
33. HasDescription
34. NoOfPopup
35. NoOfiFrame
36. HasExternalFormSubmit
37. HasSocialNet
38. HasSubmitButton
39. HasHiddenFields
40. HasPasswordField
41. Bank
42. Pay
43. Crypto
44. HasCopyrightInfo
45. NoOfImage
46. N

In [3]:
# Variance analysis of predictive features

print("VARIANCE ANALYSIS")
print("=" * 60)

# Calculate variance for all predictive features
feature_variance = df[predictive_features].var().sort_values()

# Display the features with the lowest variance
print("Features with the lowest variance:")
display(
    feature_variance.head(15).to_frame("Variance")
)

# Identify near-zero variance features
near_zero_variance = feature_variance[
    feature_variance < 0.01
]

print("\nNumber of near-zero variance features:", len(near_zero_variance))

if len(near_zero_variance) == 0:
    print("✅ No near-zero variance features found.")
else:
    print("\nNear-zero variance features:")
    display(near_zero_variance.to_frame("Variance"))

VARIANCE ANALYSIS
Features with the lowest variance:


,Variance
AmpersandDensity,0.000006
EqualsDensity,0.000009
ObfuscationRatio,0.000015
URLCharProb,0.000112
RedirectDensity,0.000323
IframeDensity,0.000457
SpecialCharDensity,0.000982
SpacialCharRatioInURL,0.001049
ImageDensity,0.001736
HasObfuscation,0.002053



Number of near-zero variance features: 17

Near-zero variance features:


,Variance
AmpersandDensity,0.000006
EqualsDensity,0.000009
ObfuscationRatio,0.000015
URLCharProb,0.000112
RedirectDensity,0.000323
IframeDensity,0.000457
SpecialCharDensity,0.000982
SpacialCharRatioInURL,0.001049
ImageDensity,0.001736
HasObfuscation,0.002053


In [4]:
# Correlation analysis between predictive features

print("FEATURE CORRELATION ANALYSIS")
print("=" * 60)

# Calculate absolute correlation matrix
correlation_matrix = df[predictive_features].corr().abs()

# Keep only the upper triangle to avoid duplicate pairs
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find highly correlated feature pairs
high_correlation_pairs = []

for column in upper_triangle.columns:
    for row in upper_triangle.index:
        correlation_value = upper_triangle.loc[row, column]

        if pd.notna(correlation_value) and correlation_value >= 0.90:
            high_correlation_pairs.append(
                (row, column, correlation_value)
            )

# Display results
if high_correlation_pairs:
    high_corr_df = pd.DataFrame(
        high_correlation_pairs,
        columns=[
            "Feature 1",
            "Feature 2",
            "Absolute Correlation"
        ]
    ).sort_values(
        by="Absolute Correlation",
        ascending=False
    )

    print("Highly correlated feature pairs (|r| >= 0.90):")
    display(high_corr_df)

    print(
        "\nNumber of highly correlated pairs:",
        len(high_corr_df)
    )
else:
    print("No feature pairs with |r| >= 0.90 were found.")

print("\nCorrelation analysis completed.")

FEATURE CORRELATION ANALYSIS
Highly correlated feature pairs (|r| >= 0.90):


,Feature 1,Feature 2,Absolute Correlation
2,DegitRatioInURL,DigitDensity,0.999998
3,SpacialCharRatioInURL,SpecialCharDensity,0.985712
1,DomainTitleMatchScore,URLTitleMatchScore,0.961008
0,URLLength,NoOfLettersInURL,0.956047
4,ExternalRefDensity,TotalReferenceDensity,0.904234
5,SelfRefDensity,TotalReferenceDensity,0.903719



Number of highly correlated pairs: 6

Correlation analysis completed.


In [5]:
# Analyze relationship between features and the target
# Feature-selection statistics are calculated using training data only

from sklearn.model_selection import train_test_split

print("FEATURE-TARGET RELATIONSHIP ANALYSIS")
print("=" * 60)

# Create URL-level groups to preserve the leakage-free split
url_groups = df[["URL", "label"]].drop_duplicates(subset="URL")

train_url_groups, test_url_groups = train_test_split(
    url_groups,
    test_size=0.20,
    random_state=42,
    stratify=url_groups["label"]
)

train_url_set = set(train_url_groups["URL"])
test_url_set = set(test_url_groups["URL"])

train_mask = df["URL"].isin(train_url_set)
test_mask = df["URL"].isin(test_url_set)

# Training data only
X_train_fs = df.loc[train_mask, predictive_features]
y_train_fs = df.loc[train_mask, target_column]

# Calculate correlation with target using training data only
target_correlations = (
    X_train_fs.join(y_train_fs)
    .corr()[target_column]
    .drop(target_column)
    .sort_values(key=abs, ascending=False)
)

print("Training records:", len(X_train_fs))
print("Testing records:", df.loc[test_mask].shape[0])

print("\nTop 20 features by absolute correlation with target:")
display(
    target_correlations.head(20).to_frame("Target Correlation")
)

print("\nFeature-target analysis completed.")

FEATURE-TARGET RELATIONSHIP ANALYSIS
Training records: 188644
Testing records: 47151

Top 20 features by absolute correlation with target:


,Target Correlation
URLSimilarityIndex,0.860190
HasSocialNet,0.784422
WebStructureScore,0.770923
HasCopyrightInfo,0.743833
HasDescription,0.690888
IsHTTPS,0.609438
DomainTitleMatchScore,0.584157
HasSubmitButton,0.579051
IsResponsive,0.548989
URLTitleMatchScore,0.538502



Feature-target analysis completed.


In [6]:
# Feature importance analysis using Random Forest

from sklearn.ensemble import RandomForestClassifier

print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

# Use a representative sample of the training data
sample_size = min(30000, len(X_train_fs))

X_train_sample = X_train_fs.sample(
    n=sample_size,
    random_state=42
)

y_train_sample = y_train_fs.loc[X_train_sample.index]

# Train Random Forest for feature importance
rf_selector = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_selector.fit(X_train_sample, y_train_sample)

# Calculate feature importance
feature_importance = pd.Series(
    rf_selector.feature_importances_,
    index=predictive_features
).sort_values(ascending=False)

print("Training sample size:", sample_size)

print("\nTop 20 most important features:")
display(
    feature_importance.head(20).to_frame("Importance")
)

print("\nFeature importance analysis completed.")

FEATURE IMPORTANCE ANALYSIS
Training sample size: 30000

Top 20 most important features:


,Importance
NoOfExternalRef,0.140835
URLSimilarityIndex,0.139816
ImageDensity,0.096514
NoOfSelfRef,0.088069
NoOfImage,0.084022
LineOfCode,0.069650
HasSocialNet,0.059274
JSDensity,0.045206
NoOfJS,0.035273
WebStructureScore,0.024940



Feature importance analysis completed.


In [7]:
# Combine feature selection criteria

print("COMBINING FEATURE SELECTION CRITERIA")
print("=" * 60)

# Create a summary table
feature_selection_summary = pd.DataFrame({
    "Variance": feature_variance,
    "TargetCorrelation": target_correlations.abs(),
    "RandomForestImportance": feature_importance
})

# Rank features according to each criterion
feature_selection_summary["CorrelationRank"] = (
    feature_selection_summary["TargetCorrelation"]
    .rank(ascending=False, method="min")
)

feature_selection_summary["ImportanceRank"] = (
    feature_selection_summary["RandomForestImportance"]
    .rank(ascending=False, method="min")
)

# Combined rank
feature_selection_summary["CombinedRank"] = (
    feature_selection_summary["CorrelationRank"]
    + feature_selection_summary["ImportanceRank"]
) / 2

# Sort by combined rank
feature_selection_summary = (
    feature_selection_summary
    .sort_values("CombinedRank")
)

print("Feature selection summary:")
display(feature_selection_summary.head(20))

print("\nTotal features analyzed:",
      len(feature_selection_summary))

COMBINING FEATURE SELECTION CRITERIA
Feature selection summary:


,Variance,TargetCorrelation,RandomForestImportance,CorrelationRank,ImportanceRank,CombinedRank
URLSimilarityIndex,8.396117e+02,0.860190,0.139816,1.0,2.0,1.5
HasSocialNet,2.481149e-01,0.784422,0.059274,2.0,7.0,4.5
WebStructureScore,4.191206e+00,0.770923,0.024940,3.0,10.0,6.5
HasDescription,2.464230e-01,0.690888,0.020536,5.0,13.0,9.0
HasCopyrightInfo,2.498261e-01,0.743833,0.014662,4.0,17.0,10.5
DomainTitleMatchScore,2.467802e+03,0.584157,0.011483,7.0,18.0,12.5
NoOfSelfRef,3.121849e+04,0.334888,0.088069,25.0,4.0,14.5
NoOfExternalRef,2.592983e+04,0.270652,0.140835,28.0,1.0,14.5
IsHTTPS,1.701240e-01,0.609438,0.004791,6.0,24.0,15.0
NoOfImage,6.306236e+03,0.284197,0.084022,26.0,5.0,15.5



Total features analyzed: 64


In [8]:
# Select final features based on combined ranking

print("FINAL FEATURE SELECTION")
print("=" * 60)

# Select top 40 features
number_of_selected_features = 40

selected_features = (
    feature_selection_summary
    .head(number_of_selected_features)
    .index
    .tolist()
)

print("Original predictive features:", len(predictive_features))
print("Selected features:", len(selected_features))
print("Features removed:", len(predictive_features) - len(selected_features))

print("\nSelected feature list:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i:2d}. {feature}")

FINAL FEATURE SELECTION
Original predictive features: 64
Selected features: 40
Features removed: 24

Selected feature list:
 1. URLSimilarityIndex
 2. HasSocialNet
 3. WebStructureScore
 4. HasDescription
 5. HasCopyrightInfo
 6. DomainTitleMatchScore
 7. NoOfSelfRef
 8. NoOfExternalRef
 9. IsHTTPS
10. NoOfImage
11. NoOfJS
12. SpecialCharDensity
13. HasSubmitButton
14. ImageDensity
15. LineOfCode
16. DegitRatioInURL
17. NoOfOtherSpecialCharsInURL
18. SelfRefDensity
19. SpacialCharRatioInURL
20. URLTitleMatchScore
21. DigitDensity
22. LetterRatioInURL
23. URLCharProb
24. NoOfLettersInURL
25. JSDensity
26. CharContinuationRate
27. IsResponsive
28. HasFavicon
29. HasTitle
30. URLLength
31. HasHiddenFields
32. DomainLength
33. NoOfDegitsInURL
34. URLDomainLengthRatio
35. NoOfiFrame
36. NoOfCSS
37. Pay
38. CSSDensity
39. TotalReferenceDensity
40. Robots


In [9]:
# Verify the selected feature set

print("SELECTED FEATURE VERIFICATION")
print("=" * 60)

# Create selected feature dataset
X_selected = df[selected_features].copy()
y_selected = df["label"].copy()

print("Selected feature dataset shape:", X_selected.shape)
print("Target shape:", y_selected.shape)

print("\nNumber of selected features:", len(selected_features))

print("\nSelected features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i:2d}. {feature}")

# Check for missing values
print("\nMissing values in selected features:",
      X_selected.isnull().sum().sum())

# Check for duplicate feature columns
print("Duplicate feature names:",
      X_selected.columns.duplicated().sum())

print("\n✅ Selected feature verification completed.")

SELECTED FEATURE VERIFICATION
Selected feature dataset shape: (235795, 40)
Target shape: (235795,)

Number of selected features: 40

Selected features:
 1. URLSimilarityIndex
 2. HasSocialNet
 3. WebStructureScore
 4. HasDescription
 5. HasCopyrightInfo
 6. DomainTitleMatchScore
 7. NoOfSelfRef
 8. NoOfExternalRef
 9. IsHTTPS
10. NoOfImage
11. NoOfJS
12. SpecialCharDensity
13. HasSubmitButton
14. ImageDensity
15. LineOfCode
16. DegitRatioInURL
17. NoOfOtherSpecialCharsInURL
18. SelfRefDensity
19. SpacialCharRatioInURL
20. URLTitleMatchScore
21. DigitDensity
22. LetterRatioInURL
23. URLCharProb
24. NoOfLettersInURL
25. JSDensity
26. CharContinuationRate
27. IsResponsive
28. HasFavicon
29. HasTitle
30. URLLength
31. HasHiddenFields
32. DomainLength
33. NoOfDegitsInURL
34. URLDomainLengthRatio
35. NoOfiFrame
36. NoOfCSS
37. Pay
38. CSSDensity
39. TotalReferenceDensity
40. Robots

Missing values in selected features: 0
Duplicate feature names: 0

✅ Selected feature verification completed.


In [10]:
# Final leakage check for the selected feature set

print("FINAL LEAKAGE CHECK")
print("=" * 60)

# Check whether target is included
target_leakage = "label" in selected_features

# Check whether metadata/text columns are included
excluded_columns = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "Title"
]

metadata_leakage = [
    feature for feature in selected_features
    if feature in excluded_columns
]

# Check for non-numerical features
non_numeric_features = [
    feature for feature in selected_features
    if not pd.api.types.is_numeric_dtype(df[feature])
]

print("Target variable included:", target_leakage)
print("Metadata/text features included:", metadata_leakage)
print("Non-numerical selected features:", non_numeric_features)

if not target_leakage:
    print("\n✅ Target variable 'label' is NOT included.")

if len(metadata_leakage) == 0:
    print("✅ Metadata/text columns are NOT included.")

if len(non_numeric_features) == 0:
    print("✅ All selected features are numerical.")

print("\nFinal selected feature count:", len(selected_features))

FINAL LEAKAGE CHECK
Target variable included: False
Metadata/text features included: []
Non-numerical selected features: []

✅ Target variable 'label' is NOT included.
✅ Metadata/text columns are NOT included.
✅ All selected features are numerical.

Final selected feature count: 40


In [11]:
# Save the final selected feature dataset

print("FINAL FEATURE SELECTION RESULTS")
print("=" * 60)

# Combine selected features with target
final_selected_df = X_selected.copy()
final_selected_df["label"] = y_selected

# Save the dataset
output_path = "../dataset/PhiUSIIL_Selected_Features.csv"

final_selected_df.to_csv(
    output_path,
    index=False
)

print("Original predictive features:", len(predictive_features))
print("Selected predictive features:", len(selected_features))
print("Features removed:", len(predictive_features) - len(selected_features))

print("\nFinal dataset shape:", final_selected_df.shape)
print("Missing values:", final_selected_df.isnull().sum().sum())
print("Duplicate rows:", final_selected_df.duplicated().sum())

print("\nTarget distribution:")
print(final_selected_df["label"].value_counts().sort_index())

print("\n" + "=" * 60)
print("✅ FEATURE SELECTION COMPLETED")
print("Selected dataset saved to:")
print(output_path)

FINAL FEATURE SELECTION RESULTS
Original predictive features: 64
Selected predictive features: 40
Features removed: 24

Final dataset shape: (235795, 41)
Missing values: 0
Duplicate rows: 1135

Target distribution:
label
0    100945
1    134850
Name: count, dtype: int64

✅ FEATURE SELECTION COMPLETED
Selected dataset saved to:
../dataset/PhiUSIIL_Selected_Features.csv
